# Nemotron v1 SFT — Colab Pro+ A100

Nemotron v1 SFT — Colab Pro+ A100 version.

> USAGE (Colab Pro+):
> 1. New Notebook → Runtime → Change runtime type → A100 (40 GB GPU,
>    High-RAM). Save & connect.
> 2. Paste this whole script into one cell.
> 3. Run All. Expected runtime: 6-10 h on A100 40 GB.
> 4. The trained LoRA adapter lands at
>    /content/drive/MyDrive/nemotron-2026/adapter_v1/. After it
>    finishes, download that folder (or upload it as a Kaggle Dataset
>    `ky7240/nemotron-v1-adapter`) and reference it from a thin
>    submission notebook (= GPU off, just writes submission.csv with
>    the adapter path) to submit to Kaggle.

ATTRIBUTION (= Apache 2.0 licensed public kernels we draw setup code
from; please credit when publishing):
- dgxchen/training-with-unsloth-to-achieve-0-85-lb (Kaggle kernel)
- konbu17/nemotron-sft-lora-with-cot (Kaggle kernel)
- huikang/end-to-end-finetuning-for-lb-0-85 (Kaggle kernel)

NOVEL CONTRIBUTION = the SFT data (data/processed/train_sft_verifier_
only.jsonl, 5418 records). It is generated by 5 deterministic Python
solvers in this repo (Roman / Physics / Unit / Cipher / Bit) producing
verifier-backed Chain-of-Thought traces. See
docs/strategy/winning-strategy.dense.md for the full rationale.

## Cell 1 — Drive mount + git clone-or-pull (hot-reload safe)

In [3]:
# Designed so re-running this cell alone refreshes the repo to the
# latest origin/main without touching cells 2 (pip install) or 3
# (model load). Cells 4 and 5 are thin wrappers around
# `colab_pipeline.py` in the repo, so any fix lands in the running
# kernel by simply re-running cells 1 -> 4 -> 5.
from google.colab import drive

# Robust mount: force_remount clears stale state from a previous
# Colab session, timeout_ms gives the OAuth popup time to complete,
# and a 1-shot retry handles transient drivefs issues that show up
# as `ValueError: mount failed`.
try:
    drive.mount("/content/drive", force_remount=True, timeout_ms=180000)
except Exception as _e:
    print(f"first mount attempt failed: {_e}; retrying after 5 s...")
    import time as _time

    _time.sleep(5)
    drive.mount("/content/drive", force_remount=True, timeout_ms=180000)

import os
import subprocess

REPO_URL = "https://github.com/ykaya-jp/NVIDIA-Nemotron-Model-Reasoning-Challenge.git"
REPO_DIR = "/content/nemotron"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    # Already cloned — fast-forward to the latest commit on origin/main.
    # `reset --hard` is safe because we never edit files inside the
    # Colab clone (the user edits happen on the host machine and reach
    # us through git).
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1",
                    "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard",
                    "origin/main"], check=True)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)
subprocess.run(["git", "log", "-1", "--oneline"])

ADAPTER_DIR = "/content/drive/MyDrive/nemotron-2026/adapter_v1"
os.makedirs(ADAPTER_DIR, exist_ok=True)

# Make the repo's Python package importable. Adding sys.path here
# (rather than in cell 4) means an `importlib.reload(...)` in cell 4
# always sees the file freshly written by the git pull above.
import sys
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))


Mounted at /content/drive
repo at /content/nemotron


## Cell 2 — Dependencies (plain transformers + peft + TRL stack)

In [4]:
# Why no Unsloth: Unsloth issue #3480 (= "Cannot load Nvidia Nemotron
# Nano models") + a meta-tensor crash inside unsloth_zoo
# fix_untrained_tokens make the Unsloth fast-path unusable on this
# specific model. The NVIDIA / DataCamp reference notebook uses plain
# transformers + peft + TRL with the version pins below; this is the
# documented working stack for Nemotron-3-Nano-30B-A3B-BF16 on an
# A100 80 GB.
# References:
# - https://www.datacamp.com/tutorial/fine-tuning-nvidia-nemotron-3-nano
# - https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16
# - https://github.com/unslothai/unsloth/issues/3480
import subprocess
import sys


def _run(cmd):
    """Run shell command, stream output to the cell so the user sees errors."""
    print(f"$ {' '.join(cmd)}")
    subprocess.run(cmd, check=True)


# Step 1: build tooling for the Mamba CUDA extensions.
_run([sys.executable, "-m", "pip", "install", "-q", "-U", "packaging", "ninja"])

# Step 2: pin torch to the CUDA 12.8 build required by Nemotron-3-Nano.
# Colab default torch is usually fine, but the cu128 wheel is the one
# NVIDIA / Unsloth document for the Mamba kernels, so we force it.
_run([sys.executable, "-m", "pip", "uninstall", "-y",
      "torch", "torchvision", "torchaudio", "triton"])
_run([sys.executable, "-m", "pip", "install", "-q",
      "torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
      "--index-url", "https://download.pytorch.org/whl/cu128"])

# Step 3: HF stack with the exact pins from the NVIDIA / DataCamp
# reference. transformers 4.56.2 + trl 0.22.2 supports
# `completion_only_loss=True` and `processing_class=tokenizer`.
# peft is pinned to <0.16 on purpose: peft 0.16+ requires
# torchao>=0.16, which in turn needs torch>=2.8 (incompatible with
# the torch 2.7.1 + CUDA 12.8 build we pinned for the Mamba kernels).
# We don't use any torchao features (no quantization), so this is the
# cleanest resolution.
_run([sys.executable, "-m", "pip", "install", "-q", "-U",
      "transformers==4.56.2", "tokenizers", "trl==0.22.2",
      "accelerate", "datasets", "peft<0.16",
      "huggingface_hub", "safetensors",
      "sentencepiece", "nbformat"])

# Step 3b: defence-in-depth — even with peft<0.16 pinned, a stale
# torchao 0.10 may live in the Colab base image and trip
# `peft.import_utils.is_torchao_available()` (it import-checks the
# minimum version *if* torchao is present). Uninstalling torchao
# makes that helper return False without raising. We never use
# torchao quantization, so removing it is safe.
_run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])

# Step 4: Mamba-SSM + causal-conv1d at the versions that ship in the
# NVIDIA reference. `--no-build-isolation` reuses the torch we just
# installed (otherwise pip spawns a clean env without torch and the
# CUDA build fails). First build ~5-10 min on A100; cached after.
_run([sys.executable, "-m", "pip", "install", "-q",
      "-U", "--no-build-isolation",
      "mamba_ssm==2.2.5", "causal_conv1d==1.5.2"])

import torch

print(
    "cuda:",
    torch.cuda.is_available(),
    "device:",
    torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "mem GB:",
    torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0,
)


$ /usr/bin/python3 -m pip install -q -U packaging ninja
$ /usr/bin/python3 -m pip uninstall -y torch torchvision torchaudio triton
$ /usr/bin/python3 -m pip install -q torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
$ /usr/bin/python3 -m pip install -q -U transformers==4.56.2 tokenizers trl==0.22.2 accelerate datasets peft<0.16 huggingface_hub safetensors sentencepiece nbformat
$ /usr/bin/python3 -m pip uninstall -y torchao
$ /usr/bin/python3 -m pip install -q -U --no-build-isolation mamba_ssm==2.2.5 causal_conv1d==1.5.2
cuda: True device: NVIDIA A100-SXM4-80GB mem GB: 79.250732421875


## Cell 3 — Load base model + tokenizer (plain transformers path)

In [5]:
# We DO NOT wrap with peft here. SFTTrainer accepts `peft_config=...`
# and applies LoRA internally — this avoids the meta-tensor edge case
# we hit when wrapping the model first and then letting TRL re-prepare
# it. attn_implementation="eager" is required: the hybrid Mamba layers
# refuse SDPA / FA-2 paths.
#
# transformers 4.56.2 workaround: AutoTokenizer.from_pretrained calls
# list_repo_templates() on `<repo>/additional_chat_templates/` and
# re-raises any 404 instead of treating "no such directory" as "no
# extra templates". Nemotron-3-Nano does not ship that directory, so
# the load dies with `EntryNotFoundError: additional_chat_templates
# does not exist on "main"`. Patch the symbol to return [] on 404.
import transformers.tokenization_utils_base
import transformers.utils.hub
from huggingface_hub.utils import EntryNotFoundError

_orig_list_repo_templates = transformers.utils.hub.list_repo_templates


def _safe_list_repo_templates(*args, **kwargs):
    try:
        return _orig_list_repo_templates(*args, **kwargs)
    except EntryNotFoundError:
        return []
    except Exception as _e:
        _msg = str(_e)
        if "404" in _msg or "Entry Not Found" in _msg or "does not exist" in _msg:
            return []
        raise


transformers.utils.hub.list_repo_templates = _safe_list_repo_templates
if hasattr(transformers.tokenization_utils_base, "list_repo_templates"):
    transformers.tokenization_utils_base.list_repo_templates = _safe_list_repo_templates
print("✓ patched list_repo_templates to tolerate missing additional_chat_templates/")

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MAX_SEQ_LEN = 2048   # = 95th-percentile of our SFT records; 4096 risks
                     #   OOM at 30B + bs1 + LoRA on A100-80 GB. Inference
                     #   side still uses 4096 (we just train on shorter
                     #   contexts).
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    dtype=torch.bfloat16,
    # device_map={"": 0} forces every parameter (including MoE
    # experts) onto GPU 0. The default `device_map="auto"` offloads
    # rarely-used experts to CPU/disk, leaving meta tensors that the
    # HF Trainer's `_move_model_to_device(model.to(cuda))` cannot
    # materialise → NotImplementedError("Cannot copy out of meta
    # tensor"). 30 B BF16 ≈ 60 GB fits on the A100-80 with room to
    # spare for LoRA + activations.
    device_map={"": 0},
    attn_implementation="eager",
)
# NOTE: do NOT call model.gradient_checkpointing_enable() here.
# SFTConfig has `gradient_checkpointing=True` by default and the
# SFTTrainer + peft_config path re-prepares the model (including
# input_require_grads wiring) when applying LoRA. Pre-enabling
# gradient checkpointing on the bare model before that happens
# desynchronises the two and can throw "RuntimeError: element 0 of
# tensors does not require grad" mid-training. Leave it to TRL.
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
print("✓ base model loaded (BF16, eager attn; grad-ckpt left to SFTTrainer)")


✓ patched list_repo_templates to tolerate missing additional_chat_templates/


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/420 [00:00<?, ?B/s]

...

model-00012-of-00013.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00013-of-00013.safetensors:   0%|          | 0.00/3.24G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/197 [00:00<?, ?B/s]

✓ base model loaded (BF16, eager attn; grad-ckpt left to SFTTrainer)


## Cell 4 — Load SFT data + chat template + upsample + length audit

In [6]:
# The body lives in colab_pipeline.build_dataset() inside the repo so
# that any future fix only requires re-running cells 1 + 4 + 5 — no
# pip-install / model-load redo. We use importlib.reload to pick up
# the latest version on every re-run.
import importlib
import nvidia_nemotron_model_reasoning_challenge.colab_pipeline as _pipeline
importlib.reload(_pipeline)

SFT_PATH = os.path.join(REPO_DIR, "data/processed/train_sft_verifier_only.jsonl")
rows = _pipeline.load_rows(SFT_PATH)
ds = _pipeline.build_dataset(rows, tokenizer, MAX_SEQ_LEN)


loaded 5418 verifier-backed records
source distribution: Counter({'solver_physics': 1597, 'solver_unit': 1594, 'solver_roman': 1576, 'solver_cipher': 605, 'solver_bit': 46})
after upsampling: 6575 records
length audit on 200-sample: max=708 p95=670 truncated@2048=0 (0.0%)


## Cell 5 — SFT training (1 epoch, completion-only loss, LoRA via TRL)

In [ ]:
# Body lives in colab_pipeline.train_lora(). Re-importing/reloading
# here picks up any fix that came in via cell 1's git pull, without
# touching the loaded model or the installed packages.
import importlib
import nvidia_nemotron_model_reasoning_challenge.colab_pipeline as _pipeline
importlib.reload(_pipeline)

trainer = _pipeline.train_lora(
    model, tokenizer, ds,
    # Drive-mounted so checkpoints survive a Colab session drop
    # (the 2026-05-14 first run lost 16 h of training because we
    # were writing to /content/, which evaporates on disconnect).
    output_dir="/content/drive/MyDrive/nemotron-2026/checkpoints_v1",
    max_seq_len=MAX_SEQ_LEN,
)


Tokenizing train dataset:   0%|          | 0/6575 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/6575 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 11}.


Step,Training Loss


## Cell 6 — Save adapter to Drive

In [ ]:
# After training the model object is a PeftModel wrapping the base.
# save_pretrained writes ONLY the LoRA adapter (~200-400 MB), which is
# what the Kaggle metric notebook expects.
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("saved adapter to", ADAPTER_DIR)
for fn in sorted(os.listdir(ADAPTER_DIR)):
    sz = os.path.getsize(os.path.join(ADAPTER_DIR, fn))
    print(f"  {fn}: {sz:,} bytes")
print()
print("Next: upload this folder to Kaggle as `ky7240/nemotron-v1-adapter`,")
print("then push the thin submission notebook (= GPU off) to Kaggle and submit.")
